# 05 · Model Comparison & Champion
Pick best by Recall → F1 → ROC-AUC, export champion. LightGBM is deployed champion.


In [ ]:
from pathlib import Path
import json, shutil, joblib
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

DATA_DIR = Path("../../data_pipeline/data/processed")
MODEL_DIR = Path("../models")
ALT = Path("../../apps/backend/ml_artifacts")
REPORT_DIR = Path("../../docs/reports")
REPORT_DIR.mkdir(parents=True, exist_ok=True)

# Load with fallbacks
fpath = MODEL_DIR / "feature_columns.json"
if not fpath.exists():
    fpath = ALT / "feature_columns.json"
with open(fpath) as f:
    feature_cols = json.load(f)

if (DATA_DIR / "X_test.csv").exists():
    X_test = pd.read_csv(DATA_DIR / "X_test.csv")[feature_cols]
    y_test = pd.read_csv(DATA_DIR / "y_test.csv")["target_risk"].astype(int)
else:
    tm = pd.read_csv(DATA_DIR / "test_matrix.csv")
    X_test, y_test = tm[feature_cols], tm["target_risk"].astype(int)

models = {}
rows = []
for name in ["Logistic Regression","Random Forest","XGBoost","LightGBM"]:
    p = MODEL_DIR / f"{name.lower().replace(' ','_')}.joblib"
    if not p.exists():
        continue
    m = joblib.load(p)
    models[name] = m
    pred, prob = m.predict(X_test), m.predict_proba(X_test)[:,1]
    rows.append({"Model":name,"Accuracy":round(accuracy_score(y_test,pred),4),"Precision":round(precision_score(y_test,pred,zero_division=0),4),"Recall":round(recall_score(y_test,pred,zero_division=0),4),"F1":round(f1_score(y_test,pred,zero_division=0),4),"ROC-AUC":round(roc_auc_score(y_test,prob),4)})

comp_df = pd.DataFrame(rows).sort_values(["Recall","F1","ROC-AUC"], ascending=False).reset_index(drop=True)
display(comp_df)


In [ ]:
# Bar chart
metrics = ["Accuracy","Precision","Recall","F1","ROC-AUC"]
x = np.arange(len(metrics))
w = 0.2
cols = ["#1b5e20","#2e7d32","#66bb6a","#8d6e63"]
fig, ax = plt.subplots(figsize=(12,6))
for i, r in comp_df.iterrows():
    ax.bar(x+i*w, [r[m] for m in metrics], w, label=r["Model"], color=cols[i%len(cols)])
ax.set_xticks(x+w*1.5)
ax.set_xticklabels(metrics)
ax.set_ylim(0,1.1)
ax.set_title("4-Model Comparison (2024-2025 Test)")
ax.set_ylabel("Score")
ax.legend()
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig(MODEL_DIR / "model_comparison.png", dpi=150, bbox_inches="tight")
plt.show()


In [ ]:
# Champion
champion_name = comp_df.iloc[0]["Model"]
champion = models[champion_name]
print(f"Champion: {champion_name} — {comp_df.iloc[0].to_dict()}")

joblib.dump(champion, MODEL_DIR / "champion_model.joblib")
if "XGBoost" in models:
    joblib.dump(models["XGBoost"], MODEL_DIR / "xgboost_agrishield_v1.joblib")
comp_df.to_csv(MODEL_DIR / "model_comparison_2015_2025.csv", index=False)

# Sync to backend
if ALT.exists():
    for f in ["champion_model.joblib","xgboost_agrishield_v1.joblib","scaler.joblib","feature_columns.json","county_ndvi_stats.csv"]:
        src = MODEL_DIR / f
        if src.exists():
            shutil.copy2(src, ALT / f)
    print(f"Synced to {ALT}")

# Report
md = f"# ML Comparison Report (2015-2025)\n\nChampion: **{champion_name}**\n\n{comp_df.to_markdown(index=False)}\n"
(REPORT_DIR / "ML_Model_Comparison_Report_2015_2025.md").write_text(md)
print(f"Saved report to {REPORT_DIR}")
